# Notebook 02 — Predictive Modeling
**MSE 433 — Olympic Medal Performance**

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.models import (
    prepare_model_data, FEATURES, TARGET_COUNT,
    fit_negbin, fit_zinb,
    predict_count, predict_zinb,
    cross_validate, model_summary_table,
    coef_table, compute_residuals, sensitivity_tornado
)

ROOT   = Path('..').resolve()
OUTPUT = ROOT / 'data' / 'output'
FIG    = ROOT / 'report' / 'figures'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Setup complete')

## 1. Load & Prepare Data

In [ ]:
summer_raw = pd.read_csv(OUTPUT / 'panel_summer.csv')
winter_raw = pd.read_csv(OUTPUT / 'panel_winter.csv')

summer = prepare_model_data(summer_raw)
winter = prepare_model_data(winter_raw)

print(f'Summer model data: {len(summer):,} rows, {summer["Year"].nunique()} Games')
print(f'Winter model data: {len(winter):,} rows, {winter["Year"].nunique()} Games')
print(f'Summer years: {sorted(summer["Year"].unique())}')
print(f'Winter years: {sorted(winter["Year"].unique())}')

## 2. Overdispersion Confirmation

In [ ]:
print('=== Overdispersion Test (variance/mean ratio) ===')
print('If ratio >> 1, Negative Binomial is preferred over Poisson')
print('KB: (B) Probability Distributions — Poisson vs. Negative Binomial\n')

for label, df in [('Summer', summer), ('Winter', winter)]:
    m = df[TARGET_COUNT].mean()
    v = df[TARGET_COUNT].var()
    print(f'{label}: mean={m:.2f}, variance={v:.2f}, ratio={v/m:.1f}x → '
          f'{"STRONGLY OVERDISPERSED → use Negative Binomial" if v/m > 3 else "mild"}')

## 3. Fit Models — Summer Games

In [ ]:
print('Fitting Negative Binomial (Summer)...')
negbin_s = fit_negbin(summer)
print('Fitting ZINB (Summer) — may take ~30 seconds...')
zinb_s = fit_zinb(summer)
print('Done.')

print(f'\nNegBin AIC: {negbin_s.aic:.1f}  |  ZINB AIC: {zinb_s.aic:.1f}')
print('Lower AIC = better distributional fit. ZINB explicitly models zero-inflation.')
print(f'ZINB converged: {zinb_s.mle_retvals["converged"]}')

In [ ]:
print('=== ZINB — Count Component Coefficients (Summer) ===')
print('These are the main model coefficients. inflate_* rows (zero-inflation equation) are excluded.')
ct_s = coef_table(zinb_s, 'zinb')
display(ct_s[['Coefficient', 'CI Lower', 'CI Upper', 'p-value', 'Significant', 'Interpretation']])

print('\n=== Negative Binomial Coefficients (Summer) — for comparison ===')
ct_negbin_s = coef_table(negbin_s, 'negbin')
display(ct_negbin_s[['Coefficient', 'CI Lower', 'CI Upper', 'p-value', 'Significant']])

## 4. Fit Models — Winter Games

In [ ]:
print('Fitting Negative Binomial (Winter)...')
negbin_w = fit_negbin(winter)
print('Fitting ZINB (Winter)...')
zinb_w = fit_zinb(winter)
print('Done.')

print(f'\nNegBin AIC: {negbin_w.aic:.1f}  |  ZINB AIC: {zinb_w.aic:.1f}')
print(f'ZINB converged: {zinb_w.mle_retvals["converged"]}')

print('\n=== ZINB — Count Component Coefficients (Winter) ===')
ct_w = coef_table(zinb_w, 'zinb')
display(ct_w[['Coefficient', 'CI Lower', 'CI Upper', 'p-value', 'Significant', 'Interpretation']])

## 5. Cross-Validation

In [ ]:
print('Running 5-fold chronological CV...')
print('Note: ZINB folds that fail to converge are automatically skipped.\n')

MODEL_SPECS = [
    ('Neg. Binomial', fit_negbin, predict_count),
    ('ZINB',          fit_zinb,   predict_zinb),
]

cv_results_summer = []
for lbl, fn, pfn in MODEL_SPECS:
    r = cross_validate(summer, fn, pfn, n_folds=5, label=lbl)
    cv_results_summer.append(r)
    print(f'  Summer {lbl}: RMSE={r["mean_rmse"]:.3f} ± {r["std_rmse"]:.3f}, MAE={r["mean_mae"]:.3f}')

print()
cv_results_winter = []
for lbl, fn, pfn in MODEL_SPECS:
    r = cross_validate(winter, fn, pfn, n_folds=5, label=lbl)
    cv_results_winter.append(r)
    print(f'  Winter {lbl}: RMSE={r["mean_rmse"]:.3f} ± {r["std_rmse"]:.3f}, MAE={r["mean_mae"]:.3f}')

In [ ]:
print('=== Table 5: Model Comparison ===')
print('\nSummer Games:')
display(model_summary_table(cv_results_summer))
print('\nWinter Games:')
display(model_summary_table(cv_results_winter))

print('\nAIC / BIC (full-data fit — lower is better):')
aic_bic = pd.DataFrame([
    {'Season': 'Summer', 'Model': 'Neg. Binomial', 'AIC': round(negbin_s.aic, 1), 'BIC': round(negbin_s.bic, 1)},
    {'Season': 'Summer', 'Model': 'ZINB',          'AIC': round(zinb_s.aic, 1),   'BIC': round(zinb_s.bic, 1)},
    {'Season': 'Winter', 'Model': 'Neg. Binomial', 'AIC': round(negbin_w.aic, 1), 'BIC': round(negbin_w.bic, 1)},
    {'Season': 'Winter', 'Model': 'ZINB',          'AIC': round(zinb_w.aic, 1),   'BIC': round(zinb_w.bic, 1)},
]).set_index(['Season', 'Model'])
display(aic_bic)

## 6. Table 6 — ZINB Coefficients (Primary Model)

In [ ]:
print('=== Table 6: ZINB Count-Component Coefficients — Summer ===')
print('KB: (B) Confidence Intervals, Hypothesis Testing; (C) Predictive Analytics')
ct_s = coef_table(zinb_s, 'zinb')
display(ct_s[['Coefficient', 'CI Lower', 'CI Upper', 'p-value', 'Significant', 'Interpretation']])

print('\n=== Table 7: ZINB Count-Component Coefficients — Winter ===')
ct_w = coef_table(zinb_w, 'zinb')
display(ct_w[['Coefficient', 'CI Lower', 'CI Upper', 'p-value', 'Significant', 'Interpretation']])

## Figure 8 — Actual vs. Predicted (ZINB)

In [ ]:
zinb_preds_s = predict_zinb(zinb_s, summer)
zinb_preds_w = predict_zinb(zinb_w, winter)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (label, df, preds, color) in zip(axes, [
    ('Summer Games', summer, zinb_preds_s, '#2196F3'),
    ('Winter Games', winter, zinb_preds_w, '#FF5722'),
]):
    actual = df[TARGET_COUNT].values
    mask   = actual > 0
    ax.scatter(preds[mask],  actual[mask],  alpha=0.45, s=16, color=color, label='Medal-winning')
    ax.scatter(preds[~mask], actual[~mask], alpha=0.12, s=8,  color='grey', label='Zero-medal')
    lim = max(actual.max(), preds.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1.5, label='Perfect prediction (45°)')
    ax.set_xlabel('Predicted Medals (ZINB)')
    ax.set_ylabel('Actual Medals')
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=8)

fig.suptitle('Figure 8: Actual vs. Predicted Medal Counts (ZINB)\n'
             'Points near the 45° line = accurate predictions; outliers = over/underperformers\n'
             'KB: (C) Data Science — Predictive Analytics, Residual Analysis',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG / 'fig8_actual_vs_predicted.png', bbox_inches='tight')
plt.show()
print('Saved Figure 8')

## Figure 9 — Residuals: Over/Underperformers (ZINB)

In [ ]:
res_summer = compute_residuals(summer, zinb_s, predict_zinb, top_n=10)
res_winter  = compute_residuals(winter,  zinb_w, predict_zinb, top_n=10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (label, res, color) in zip(axes, [
    ('Summer Games', res_summer, '#2196F3'),
    ('Winter Games', res_winter,  '#FF5722'),
]):
    top_over  = res.head(10)
    top_under = res.tail(10)
    plot_df   = pd.concat([top_over, top_under]).sort_values('residual')
    bar_colors = ['#4CAF50' if v > 0 else '#F44336' for v in plot_df['residual']]
    ax.barh(plot_df.index, plot_df['residual'], color=bar_colors, alpha=0.8)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Mean Residual (Actual − Predicted)')
    ax.set_title(f'{label}\nGreen = consistent overperformers | Red = underperformers', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)

fig.suptitle('Figure 9: Systematic Over/Underperformers — Mean ZINB Residuals (1960–2016)\n'
             'Mean residual per country across all Games in the panel\n'
             'KB: (C) Data Science — Residual Plots, Predictive Analytics',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG / 'fig9_residuals.png', bbox_inches='tight')
plt.show()
print('Saved Figure 9')

## Figure 10 — Sensitivity Tornado (ZINB)

In [ ]:
# Baseline = median Summer country-game profile
baseline_summer = summer[FEATURES].median()
tornado_s = sensitivity_tornado(zinb_s, predict_zinb, baseline_summer, delta=0.10)

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#4CAF50' if v > 0 else '#F44336' for v in tornado_s['Change in Predicted Medals']]
labels = tornado_s['Feature'] + ' (' + tornado_s['Direction'] + ')'
ax.barh(labels, tornado_s['Change in Predicted Medals'], color=bar_colors, alpha=0.8)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Change in Predicted Medals')
ax.set_title('Figure 10: Sensitivity Tornado — ±10% Input Change (Summer ZINB)\n'
             'Baseline = median Summer country-game; shows which levers matter most\n'
             'KB: (A) Sensitivity Analysis; (C) Predictive Analytics',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG / 'fig10_sensitivity_tornado.png', bbox_inches='tight')
plt.show()
print('Saved Figure 10')

print('\n=== Sensitivity summary ===')
display(tornado_s[['Feature', 'Direction', 'Change in Predicted Medals']].round(3))

## Key Model Stats

In [ ]:
print('=== KEY STATS ===')
for label, negbin, zinb, df in [('Summer', negbin_s, zinb_s, summer), ('Winter', negbin_w, zinb_w, winter)]:
    negbin_preds = predict_count(negbin, df)
    zinb_preds   = predict_zinb(zinb, df)
    actual       = df[TARGET_COUNT].values

    def safe_rmse(preds, actual):
        if not np.all(np.isfinite(preds)): return float('nan')
        return np.sqrt(((actual - preds)**2).mean())

    print(f'\n{label}:')
    print(f'  NegBin — AIC={negbin.aic:.1f}, BIC={negbin.bic:.1f}, In-sample RMSE={safe_rmse(negbin_preds, actual):.2f}')
    print(f'  ZINB   — AIC={zinb.aic:.1f},   BIC={zinb.bic:.1f},   In-sample RMSE={safe_rmse(zinb_preds, actual):.2f}')

    params = zinb.params
    conf   = zinb.conf_int()
    for f in ['log_delegation_size', 'log_gdp_per_capita', 'is_host', 'female_ratio', 'sport_hhi']:
        if f in params.index:
            coef  = params[f]
            ci_lo = conf.loc[f, 0]
            ci_hi = conf.loc[f, 1]
            pval  = zinb.pvalues[f]
            sig   = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ''))
            print(f'  {f}: coef={coef:.3f} [{ci_lo:.3f}, {ci_hi:.3f}] {sig}')

In [ ]:
import pickle

with open(OUTPUT / 'negbin_summer.pkl', 'wb') as f:
    pickle.dump(negbin_s, f)
with open(OUTPUT / 'negbin_winter.pkl', 'wb') as f:
    pickle.dump(negbin_w, f)
with open(OUTPUT / 'zinb_summer.pkl', 'wb') as f:
    pickle.dump(zinb_s, f)
with open(OUTPUT / 'zinb_winter.pkl', 'wb') as f:
    pickle.dump(zinb_w, f)

print('Saved: negbin_summer.pkl, negbin_winter.pkl')
print('Saved: zinb_summer.pkl, zinb_winter.pkl')
print(f'All models in: {OUTPUT}')